In [1]:
from datasets import load_dataset

dataset = load_dataset(
    "abisee/cnn_dailymail",
    "3.0.0"
)

c:\Users\omm47\PythonEnvs\ai_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [3]:
print(f"Features: {dataset['train'].column_names}")

Features: ['article', 'highlights', 'id']


In [4]:
sample = dataset["train"][0]

In [5]:
sample.keys()

dict_keys(['article', 'highlights', 'id'])

In [6]:
print(f"""Article (excerpt of 500 characters, total length: {len(sample["article"])})""")
# let's 500 characters of article
print(sample["article"][:500])

Article (excerpt of 500 characters, total length: 2527)
LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as s


In [7]:
print(f"Summary (length: {len(sample["highlights"])}):")
print(sample["highlights"])

Summary (length: 217):
Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .


## **Text Summarization Pipelines**
- comparing different models over the same input text

In [8]:
sample_text = dataset["train"][1]["article"][:2000] # would be used as input text for models
summaries = {} # the generated summaries by the model would be stored in this dict

In [9]:
import nltk
from nltk.tokenize import sent_tokenize
nltk.download("punkt_tab")
string = "The U.S. is a country. The U.N. is an organization."
sent_tokenize(string)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


['The U.S. is a country.', 'The U.N. is an organization.']

### **Baseline Summarization**

- This is one of the simplest baselines in NLP, just choose the first 3 sentences

In [10]:
def three_sentence_baseline(text):
  return "\n".join(sent_tokenize(text)[:3])

In [11]:
summaries["baseline"] = three_sentence_baseline(sample_text)

## **GPT-2**

In [12]:
from transformers import pipeline, set_seed
set_seed(42)
pipe = pipeline("text-generation", model="gpt2-xl")
gpt2_query = sample_text + "\nTL;DR:\n"
pipe_out = pipe(gpt2_query, max_length=512, clean_up_tokenization_spaces=True)
summaries["gpt2"] = "\n".join(sent_tokenize(pipe_out[0]["generated_text"][len(gpt2_query) :]))


config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 6.43GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [14]:
summaries["gpt2"]

"The mentally ill aren't being treated properly and are being housed in an environment that is not conducive to their care.\nThe inmates are living on the ninth floor of the jail, a place where there are no beds and no proper ventilation.\nThe mentally ill inmates are not getting any help and are instead getting worse.\nThe mentally ill inmates are often violent at times, but it could be for a variety of reasons.\nThe inmates are housed in a facility that is not designed for the mentally ill and is inhumane.\nThe number of mentally ill people in the country is increasing and not being adequately treated.\nThe mentally ill are dying in jail and the number of mentally ill people dying is rising.\nThe mentally ill are being put into jails and prisons that are unsafe to them and they are dying in jail.\nThe mentally ill are suffering in jail and hospitals.\nThe number of mentally ill people in jails and prisons is increasing but they are being neglected.\nThe mentally ill are dying in jail

In [27]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("t5-large")
model = AutoModelForSeq2SeqLM.from_pretrained("t5-large")

inputs = tokenizer(
    "summarize: " + sample_text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

outputs = model.generate(
    **inputs,
    max_new_tokens=512
)

summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
summaries["t5"] = "\n".join(sent_tokenize(summary))

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

In [38]:
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

inputs = tokenizer(
    sample_text,          # No "summarize:" prefix for BART
    return_tensors="pt",
    truncation=True,
    max_length=1024       # BART accepts up to 1024 input tokens
)

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    num_beams=4,
    early_stopping=True
)
summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
# summaries["bart"] = "\n".join(sent_tokenize(summary))

AttributeError: 'list' object has no attribute 'keys'